In [ ]:
from dataclasses import dataclass
from typing import List, Tuple
import math

@dataclass
class CardState:
    ease_factor: float = 2.5
    interval: float = 0
    step_index: int = 0
    lapses: int = 0

@dataclass
class DeckOptions:
    learning_steps: List[float] = None
    graduating_interval: float = 1
    easy_interval: float = 4
    interval_modifier: float = 0.8  # Reduced from 1.0
    maximum_interval: float = 36500
    easy_bonus: float = 1.2  # Reduced from 1.3
    hard_factor: float = 1.1  # Reduced from 1.2
    lapse_factor: float = 0.5  # New parameter for handling lapses
    ease_decay: float = 0.15  # New parameter for ease factor decay on wrong answers
    ease_boost: float = 0.1  # New parameter for ease factor boost on correct answers

def calculate_next_interval(card: CardState, quality: int, elapsed_days: float, options: DeckOptions) -> Tuple[CardState, float]:
    """
    Calculate the next interval for a card based on the answer quality and elapsed time.
    
    quality: 
    0 - Again (Fail)
    1 - Hard
    2 - Good
    3 - Easy
    
    Returns: (updated CardState, next_interval)
    """
    if options.learning_steps is None:
        options.learning_steps = [1, 10]

    if card.interval == 0:  # Card is in learning
        if quality == 0:  # Again
            card.step_index = 0
            return card, options.learning_steps[0]
        elif quality == 3:  # Easy
            card.interval = options.easy_interval
            return card, card.interval
        else:  # Hard or Good
            card.step_index += 1
            if card.step_index < len(options.learning_steps):
                return card, options.learning_steps[card.step_index]
            else:
                card.interval = options.graduating_interval
                return card, card.interval
    
    # Card is in review
    if quality == 0:  # Again
        card.lapses += 1
        card.interval *= options.lapse_factor
        card.ease_factor = max(1.3, card.ease_factor - options.ease_decay)
    elif quality == 1:  # Hard
        card.interval *= options.hard_factor
        card.ease_factor = max(1.3, card.ease_factor - options.ease_decay / 2)
    elif quality == 2:  # Good
        card.interval *= card.ease_factor
    elif quality == 3:  # Easy
        card.interval *= card.ease_factor * options.easy_bonus
        card.ease_factor = min(2.8, card.ease_factor + options.ease_boost)

    # Factor in the actual elapsed time
    card.interval = max(card.interval, elapsed_days)
    
    # Apply interval modifier and constrain to maximum
    card.interval *= options.interval_modifier
    card.interval = min(card.interval, options.maximum_interval)
    
    return card, math.ceil(card.interval)

# Example usage:
if __name__ == "__main__":
    options = DeckOptions()
    card = CardState()

    # Simulate a series of reviews
    reviews = [3, 2, 2, 1, 2, 3, 0, 2, 2, 3]
    for review_count, quality in enumerate(reviews, 1):
        if card.interval == 0:
            elapsed_days = 0  # For cards in learning, assume no delay
        else:
            elapsed_days = card.interval * 1.1  # Simulate answering slightly late
        
        card, next_interval = calculate_next_interval(card, quality, elapsed_days, options)
        print(f"Review {review_count}: Quality: {quality}, Next Interval: {next_interval:.1f} days, Ease Factor: {card.ease_factor:.2f}, Lapses: {card.lapses}")

In [ ]:
import random
from datetime import datetime, timedelta

class Card:
    def __init__(self, id, front, back):
        self.id = id
        self.front = front
        self.back = back
        self.ease_factor = 2.5
        self.interval = 0
        self.due = datetime.now()
        self.state = "New"

class SpacedRepetitionSystem:
    def __init__(self):
        self.cards = []

    def add_card(self, front, back):
        card_id = len(self.cards) + 1
        self.cards.append(Card(card_id, front, back))

    def get_due_cards(self):
        return [card for card in self.cards if card.due <= datetime.now()]

    def answer_card(self, card, quality):
        if card.state == "New" or card.state == "Learning":
            self._handle_learning(card, quality)
        else:
            self._handle_review(card, quality)

    def _handle_learning(self, card, quality):
        if quality >= 3:  # Good or Easy
            card.state = "Review"
            card.interval = 1
            card.due = datetime.now() + timedelta(days=1)
        else:  # Again or Hard
            card.state = "Learning"
            card.due = datetime.now() + timedelta(minutes=10)

    def _handle_review(self, card, quality):
        if quality == 1:  # Again
            card.interval = max(1, int(card.interval * 0.5))
            card.ease_factor = max(1.3, card.ease_factor - 0.2)
            card.due = datetime.now() + timedelta(days=1)
        else:
            if quality == 2:  # Hard
                card.interval = int(card.interval * 1.2)
                card.ease_factor = max(1.3, card.ease_factor - 0.15)
            elif quality == 3:  # Good
                card.interval = int(card.interval * card.ease_factor)
            elif quality == 4:  # Easy
                card.interval = int(card.interval * card.ease_factor * 1.3)
                card.ease_factor = min(2.5, card.ease_factor + 0.15)

            card.due = datetime.now() + timedelta(days=card.interval)

        card.interval = min(card.interval, 365)  # Cap interval at 1 year

    def review_card(self, card):
        print(f"Card front: {card.front}")
        input("Press Enter to see the back...")
        print(f"Card back: {card.back}")
        while True:
            quality = input("Rate your recall (1: Again, 2: Hard, 3: Good, 4: Easy): ")
            if quality in ['1', '2', '3', '4']:
                return int(quality)
            print("Invalid input. Please enter 1, 2, 3, or 4.")

# Usage example
srs = SpacedRepetitionSystem()

# Add some sample cards
srs.add_card("What is the capital of France?", "Paris")
srs.add_card("What is 2 + 2?", "4")
srs.add_card("Who wrote 'Romeo and Juliet'?", "William Shakespeare")

# Simulation of reviewing cards over time
for day in range(30):
    print(f"\nDay {day + 1}")
    due_cards = srs.get_due_cards()
    if not due_cards:
        print("No cards due today.")
        continue

    for card in due_cards:
        print(f"\nReviewing card {card.id}")
        quality = srs.review_card(card)
        srs.answer_card(card, quality)
        print(f"Next review in {card.interval} days")



    # Simulate time passing
    for card in srs.cards:
        card.due -= timedelta(days=1)

In [ ]:
from fsrs import *

f = FSRS()

In [ ]:
card = Card()

In [ ]:
rating = Rating.Good

card, review_log = f.review_card(card, rating)

In [ ]:
from datetime import datetime, timezone

due = card.due

# how much time between when the card is due and now
time_delta = due - datetime.now(timezone.utc)

print(f"Card due: at {repr(due)}")
print(f"Card due in {time_delta.seconds} seconds")

In [ ]:
from datetime import datetime, timezone

# custom review time (must be UTC)
review_time = datetime(2024, 7, 13, 20, 7, 56, 150101, tzinfo=timezone.utc)

scheduling_cards = f.repeat(card, review_time)

# can get updated cards for each possible rating
card_Again = scheduling_cards[Rating.Again].card
card_Hard = scheduling_cards[Rating.Hard].card
card_Good = scheduling_cards[Rating.Good].card
card_Easy = scheduling_cards[Rating.Easy].card

# get next review interval for each rating
scheduled_days_Again = card_Again.scheduled_days
scheduled_days_Hard = card_Hard.scheduled_days
scheduled_days_Good = card_Good.scheduled_days
scheduled_days_Easy = card_Easy.scheduled_days

# choose a rating and update the card
rating = Rating.Good
card = scheduling_cards[rating].card

# get the corresponding review log for the review
review_log = scheduling_cards[rating].review_log

In [ ]:
# serialize before storage
card_dict = card.to_dict()
review_log_dict = review_log.to_dict()

# deserialize from dict
new_card = Card.from_dict(card_dict)
new_review_log = ReviewLog.from_dict(review_log_dict)

In [ ]:
def cefr_to_lexile_diagnostic(cefr):
  cefr_to_lexile = {
    "A1_1": (178, 1),
      "A1_2": (356,2),
      "A1_3": (534, 3),
      "A2_1": (620, 4),
      "A2_2": (706, 5),
      "A2_3": (792,6),
      "B1_1": (888, 7),
      "B1_2": (983, 8 ),
      "B1_3": (1078, 9),
      "B2_1": (1153, 10),
      "B2_2": (1228, 11),
      "B2_3": (1303, 12),
      "C1_1": (1325, 13),
      "C1_2": (1393, 14),
      "C1_3": (1460, 15),
      "C2_1": (1465, 16),
      "C2_2": (1565, 17),
      "C2_3": (1665, 18),
  }
  return cefr_to_lexile[cefr]

In [ ]:
def rank_to_lexile(rank): 
  cefr_to_lexile= {
    1: "A1_1",
      2:"A1_2",
     3: "A1_3",
      4:"A2_1",
     5: "A2_2",
     6 :"A2_3",
      7:"B1_1",
     8 :"B1_2",
      9:"B1_3",
      10:"B2_1",
     11 :"B2_2",
     12 :"B2_3",
     13 :"C1_1",
     14 :"C1_2",
     15: "C1_3",
      16:"C2_1",
      17:"C2_2",
      18:"C2_3", 
  }
  return cefr_to_lexile[rank]

In [ ]:
object = {}
lexile = cefr_to_lexile_diagnostic("B1_3")[0]
lexile

In [ ]:
class LexileAlgorithm:
  def __init__(self, score, first_time, past_cefr_level): 
    self.score = score
    self.first_time = first_time
    self.past_cefr_level = past_cefr_level
  def BinarySearch(self): 
    if self.first_time:
      low = 1
      high = 18
      object["low"] = low
      object["high"] = high
      
      mid = cefr_to_lexile_diagnostic(past_cefr_level)[1]
    else: 
      low = object["low"]
      high = object["high"]
      mid = (low + high) // 2
    # TODO add a way to test the hypothesis that teachers have no idea about their students regarding their level...
    while low <= high:
        if self.score >= 80:
          # If student passes this level, try a higher one
          low = mid + 1
          global lexile
          lexile = rank_to_lexile(mid)
          object["low"] = low
          break
        else:
          # If student fails this level, try a lower one
          high = mid - 1
          object["high"] = high
          break
    if low > high:
      return True
    else: 
      return False

In [ ]:
LexileAlgorithm(50, True).BinarySearch()

In [ ]:
print(object, lexile)

In [ ]:
LexileAlgorithm(40, False).BinarySearch()

In [ ]:
print(object, lexile)

In [ ]:
LexileAlgorithm(40, False).BinarySearch()

In [ ]:
print(object, lexile)

In [ ]:
LexileAlgorithm(40, False).BinarySearch()

In [ ]:
import numpy as np
bools = [True, False, False, False, False, False, False, False, False]

past_cefrs = ["A1_1",
      "A1_2",
     "A1_3",
      "A2_1",
     "A2_2",
     "A2_3",
      "B1_1",
     "B1_2",
      "B1_3",
      "B2_1",
     "B2_2",
    "B2_3",
     "C1_1",
     "C1_2",
     "C1_3",
     "C2_1",
      "C2_2",
      "C2_3", ]
for past_cefr in past_cefrs:
  bools_iter = iter(bools)
  scores = [np.random.randint(0,100) for x in range(0,7)]
  print(scores)
  for index, score in enumerate(scores): 
    done = LexileAlgorithm(score, next(bools_iter)).BinarySearch()
    print(index)
    if done: 
      break